In [ ]:
# ============================================================
# CBM IMPLEMENTATION
# ============================================================

import os
import random
import warnings
import builtins

import numpy as np
import pandas as pd
from PIL import Image

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

original_print = builtins.print

# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(42)

# ============================================================
# CONFIG
# ============================================================

class Config:

    META_DIR   = "meta"
    IMAGES_DIR = "images"
    OUTPUT_DIR = "models"
    SPLITS_DIR = "splits"

    TUNABLE_LAYERS    = 75
    CONCEPT_WEIGHTING = True
    BATCH_SIZE        = 16
    EPOCHS            = 20
    IMG_SIZE          = 224
    LEARNING_RATE     = 5e-4
    ETA_MIN           = 1e-5
    LAMBDA1           = 1.0
    LAMBDA2           = 1.0

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    CONCEPT_COLUMNS = [
        "pigment_network", "streaks", "pigmentation",
        "regression_structures", "dots_and_globules",
        "blue_whitish_veil", "vascular_structures"
    ]
    TARGET_COLUMN = "diagnosis"

CFG = Config()
os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)

# ============================================================
# CONCEPT ENCODERS
# ============================================================

def build_global_concept_encoders(full_meta_df):
    concept_info, label_encoders = {}, {}
    for c in CFG.CONCEPT_COLUMNS:
        le = LabelEncoder()
        le.fit(full_meta_df[c].astype(str).fillna("unknown").values)
        label_encoders[c] = le
        concept_info[c]   = {"n_classes": len(le.classes_)}
    return concept_info, label_encoders

# ============================================================
# BACKBONE
# ============================================================

def get_backbone(name):
    if name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        features, feature_dim = m.features, 1280
    elif name == "efficientnet_b1":
        m = models.efficientnet_b1(weights=models.EfficientNet_B1_Weights.DEFAULT)
        features, feature_dim = m.features, 1280
    elif name == "efficientnet_b2":
        m = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.DEFAULT)
        features, feature_dim = m.features, 1408
    elif name == "efficientnet_b3":
        m = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
        features, feature_dim = m.features, 1536
    elif name == "efficientnet_b4":
        m = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.DEFAULT)
        features, feature_dim = m.features, 1792
    elif name == "efficientnet_b5":
        m = models.efficientnet_b5(weights=models.EfficientNet_B5_Weights.DEFAULT)
        features, feature_dim = m.features, 2048
    elif name == "efficientnet_b6":
        m = models.efficientnet_b6(weights=models.EfficientNet_B6_Weights.DEFAULT)
        features, feature_dim = m.features, 2304
    elif name == "efficientnet_b7":
        m = models.efficientnet_b7(weights=models.EfficientNet_B7_Weights.DEFAULT)
        features, feature_dim = m.features, 2560
    elif name == "densenet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        features, feature_dim = m.features, 1024
    elif name == "densenet161":
        m = models.densenet161(weights=models.DenseNet161_Weights.DEFAULT)
        features, feature_dim = m.features, 2208
    elif name == "densenet169":
        m = models.densenet169(weights=models.DenseNet169_Weights.DEFAULT)
        features, feature_dim = m.features, 1664
    elif name == "densenet201":
        m = models.densenet201(weights=models.DenseNet201_Weights.DEFAULT)
        features, feature_dim = m.features, 1920
    elif name == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        features, feature_dim = nn.Sequential(*list(m.children())[:-2]), 512
    elif name == "resnet34":
        m = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        features, feature_dim = nn.Sequential(*list(m.children())[:-2]), 512
    elif name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        features, feature_dim = nn.Sequential(*list(m.children())[:-2]), 2048
    elif name == "resnet101":
        m = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
        features, feature_dim = nn.Sequential(*list(m.children())[:-2]), 2048
    elif name == "resnet152":
        m = models.resnet152(weights=models.ResNet152_Weights.DEFAULT)
        features, feature_dim = nn.Sequential(*list(m.children())[:-2]), 2048
    elif name == "wide_resnet50_2":
        m = models.wide_resnet50_2(weights=models.Wide_ResNet50_2_Weights.DEFAULT)
        features, feature_dim = nn.Sequential(*list(m.children())[:-2]), 2048
    elif name == "wide_resnet101_2":
        m = models.wide_resnet101_2(weights=models.Wide_ResNet101_2_Weights.DEFAULT)
        features, feature_dim = nn.Sequential(*list(m.children())[:-2]), 2048
    else:
        raise ValueError(f"Unknown backbone: {name!r}")

    total        = len(features)
    freeze_up_to = int(round(total * (100 - CFG.TUNABLE_LAYERS) / 100))
    for i, layer in enumerate(features):
        for p in layer.parameters():
            p.requires_grad = (i >= freeze_up_to)

    return features, feature_dim

# ============================================================
# DATASET
# ============================================================

class Derm7ptDataset(Dataset):
    def __init__(self, df, images_dir, concept_info, label_encoders, transform=None):
        self.df             = df.reset_index(drop=True)
        self.images_dir     = images_dir
        self.transform      = transform
        self.concept_info   = concept_info
        self.label_encoders = label_encoders

        self.image_paths = {}
        for root, _, files in os.walk(images_dir):
            for f in files:
                if f.lower().endswith((".jpg", ".jpeg", ".png")):
                    full = os.path.join(root, f)
                    rel  = os.path.relpath(full, images_dir).replace("\\", "/").lower()
                    self.image_paths[rel] = full

        for c in CFG.CONCEPT_COLUMNS:
            self.df[c] = label_encoders[c].transform(
                df[c].astype(str).fillna("unknown").values)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        rel_path = str(row["derm"]).replace("\\", "/").lower()
        if rel_path not in self.image_paths:
            raise FileNotFoundError(f"Image not found: {rel_path}")
        image = Image.open(self.image_paths[rel_path]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        concepts = torch.tensor(
            [int(row[c]) for c in CFG.CONCEPT_COLUMNS], dtype=torch.long
        )
        label = torch.tensor(int(row[CFG.TARGET_COLUMN]), dtype=torch.long)
        return image, concepts, label

# ============================================================
# TRANSFORMS
# ============================================================

def get_train_transforms():
    return transforms.Compose([
        transforms.RandomResizedCrop(CFG.IMG_SIZE, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=90),
        transforms.RandomApply([
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), shear=15)
        ], p=0.5),
        transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
        transforms.RandAugment(num_ops=2, magnitude=9),
        transforms.RandomChoice([
            transforms.RandomAdjustSharpness(sharpness_factor=2.0),
            transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
            transforms.Lambda(lambda x: x),
        ]),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


def get_val_transforms():
    return transforms.Compose([
        transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

# ============================================================
# MODEL
# ============================================================

class ConceptBottleneckModel(nn.Module):
    def __init__(self, backbone_name, concept_info, num_classes=2):
        super().__init__()
        self.feature_extractor, self.feature_dim = get_backbone(backbone_name)
        self.pool          = nn.AdaptiveAvgPool2d(1)
        self.concept_info  = concept_info
        self.concept_heads = nn.ModuleDict({
            c: nn.Linear(self.feature_dim, concept_info[c]["n_classes"])
            for c in CFG.CONCEPT_COLUMNS
        })
        total_concept_dim = sum(
            concept_info[c]["n_classes"] for c in CFG.CONCEPT_COLUMNS
        )
        self.classifier = nn.Sequential(
            nn.Linear(total_concept_dim, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        feats = self.feature_extractor(x)
        feats = self.pool(feats).view(feats.size(0), -1)
        concept_logits, one_hot_concepts = [], []
        for c in CFG.CONCEPT_COLUMNS:
            logits = self.concept_heads[c](feats)
            concept_logits.append(logits)
            hard = F.one_hot(
                torch.argmax(logits, dim=1),
                num_classes=self.concept_info[c]["n_classes"],
            ).float()
            one_hot_concepts.append(hard.detach())
        logits_y = self.classifier(torch.cat(one_hot_concepts, dim=1))
        return concept_logits, logits_y

# ============================================================
# LOSS
# ============================================================

class CBMLoss(nn.Module):
    def __init__(self, class_weights_label, concept_class_weights,
                 use_concept_weights=True):
        super().__init__()
        self.label_weight        = class_weights_label
        self.concept_weights     = concept_class_weights
        self.use_concept_weights = use_concept_weights

    def forward(self, concept_preds, concept_true, logits_y, labels):
        loss_c = torch.stack([
            F.cross_entropy(
                concept_preds[i],
                concept_true[:, i],
                weight=(self.concept_weights[c] if self.use_concept_weights else None),
            )
            for i, c in enumerate(CFG.CONCEPT_COLUMNS)
        ]).mean()

        loss_y = F.cross_entropy(
            logits_y, labels, weight=self.label_weight
        )

        return CFG.LAMBDA1 * loss_c + CFG.LAMBDA2 * loss_y

# ============================================================
# TRAIN / EVAL
# ============================================================

def train_epoch(model, loader, optimizer, criterion):

    model.train()
    total_loss = 0.0

    for imgs, concepts, labels in loader:
        imgs     = imgs.to(CFG.DEVICE)
        concepts = concepts.to(CFG.DEVICE)
        labels   = labels.to(CFG.DEVICE)
        optimizer.zero_grad()
        concept_preds, logits_y = model(imgs)
        loss = criterion(concept_preds, concepts, logits_y, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)

def evaluate(model, loader):

    model.eval()
    all_preds, all_labels = [], []
    c_preds = [[] for _ in CFG.CONCEPT_COLUMNS]
    c_true  = [[] for _ in CFG.CONCEPT_COLUMNS]

    with torch.no_grad():
        for imgs, concepts, labels in loader:
            imgs     = imgs.to(CFG.DEVICE)
            concepts = concepts.to(CFG.DEVICE)
            labels   = labels.to(CFG.DEVICE)
            concept_logits, logits_y = model(imgs)
            all_preds.extend(logits_y.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            for i, lg in enumerate(concept_logits):
                c_preds[i].extend(lg.argmax(1).cpu().numpy())
                c_true[i].extend(concepts[:, i].cpu().numpy())

    acc   = accuracy_score(all_labels, all_preds)
    f1    = f1_score(all_labels, all_preds, average="macro")
    c_acc = np.mean([
        accuracy_score(c_true[i], c_preds[i])
        for i in range(len(CFG.CONCEPT_COLUMNS))])
    c_f1  = np.mean([
        f1_score(c_true[i], c_preds[i], average="macro", zero_division=0)
        for i in range(len(CFG.CONCEPT_COLUMNS))])
    return acc, f1, c_acc, c_f1

print("CBM implementation ready.")
print(f"Device : {CFG.DEVICE}")

In [ ]:
# ============================================================
# LOAD AND RUN
# ============================================================

import sys
import copy
import torch.optim as optim
from torch.utils.data import DataLoader

original_print = builtins.print

# ============================================================
# BACKBONE
# ============================================================

BACKBONE_LIST = [
    "efficientnet_b5",
]


def run_backbone(backbone_name, run_id, meta, concept_info, label_encoders):
    
    set_seed(42)

    tag = (
        f"run{run_id}"
        f"_{backbone_name}"
        f"_tl{CFG.TUNABLE_LAYERS}pct"
        f"_bs{CFG.BATCH_SIZE}"
    )

    log_path = os.path.join(CFG.OUTPUT_DIR, f"{tag}.txt")
    log_file = open(log_path, "w", encoding="utf-8")

    def pf(*args, **kwargs):
        original_print(*args, **kwargs)
        sys.stdout.flush()
        log_file.write(" ".join(str(a) for a in args) + kwargs.get("end", "\n"))
        log_file.flush()

    try:

        pf(f"  tunable_layers   : {CFG.TUNABLE_LAYERS}%")
        pf(f"  optimizer        : Adam")
        pf(f"  instance weights : uniform")
        pf(f"  batch_size       : {CFG.BATCH_SIZE}")
        pf(f"  concept_weighting: {CFG.CONCEPT_WEIGHTING}")
        pf(f"  epochs           : {CFG.EPOCHS}")
        pf()

        # ---- filtered splits ----
        train_idx = pd.read_csv(os.path.join(CFG.SPLITS_DIR, "symmetric_train_indexes_filtered.csv"))["indexes"].values
        valid_idx = pd.read_csv(os.path.join(CFG.SPLITS_DIR, "symmetric_valid_indexes_filtered.csv"))["indexes"].values
        test_idx  = pd.read_csv(os.path.join(CFG.SPLITS_DIR, "symmetric_test_indexes_filtered.csv"))["indexes"].values

        train_df = meta.iloc[train_idx].copy()
        valid_df = meta.iloc[valid_idx].copy()
        test_df  = meta.iloc[test_idx].copy()

        # ---- class weights ----
        y_train = train_df[CFG.TARGET_COLUMN].values
        present = np.unique(y_train)
        w_arr   = compute_class_weight("balanced", classes=present, y=y_train)
        fw      = np.ones(2)
        for cls, w in zip(present, w_arr):
            fw[cls] = w
        label_weights = torch.tensor(fw, dtype=torch.float32).to(CFG.DEVICE)

        concept_class_weights = {}
        for c in CFG.CONCEPT_COLUMNS:
            le    = label_encoders[c]
            y_enc = le.transform(train_df[c].astype(str).fillna("unknown"))
            pres  = np.unique(y_enc)
            w_arr = compute_class_weight("balanced", classes=pres, y=y_enc)
            fw    = np.ones(len(le.classes_))
            for cls, w in zip(pres, w_arr):
                fw[cls] = w
            concept_class_weights[c] = torch.tensor(fw, dtype=torch.float32).to(CFG.DEVICE)

        # ---- data loaders ----
        g = torch.Generator()
        g.manual_seed(42)

        train_loader = DataLoader(
            Derm7ptDataset(train_df, CFG.IMAGES_DIR, concept_info, label_encoders, get_train_transforms()),
            batch_size=CFG.BATCH_SIZE, shuffle=True,
            num_workers=0, generator=g, pin_memory=True)

        val_loader = DataLoader(
            Derm7ptDataset(valid_df, CFG.IMAGES_DIR, concept_info, label_encoders, get_val_transforms()),
            batch_size=CFG.BATCH_SIZE, num_workers=2, pin_memory=True)

        test_loader = DataLoader(
            Derm7ptDataset(test_df, CFG.IMAGES_DIR, concept_info, label_encoders, get_val_transforms()),
            batch_size=CFG.BATCH_SIZE, num_workers=2, pin_memory=True)

        # ---- model, loss, optimizer ----
        model = ConceptBottleneckModel(backbone_name, concept_info).to(CFG.DEVICE)
        total_p     = sum(p.numel() for p in model.parameters())
        trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
        pf(f"Params: total={total_p:,} | trainable={trainable_p:,} | frozen={total_p - trainable_p:,}")

        criterion = CBMLoss(label_weights, concept_class_weights, use_concept_weights=CFG.CONCEPT_WEIGHTING)

        optimizer = optim.Adam(model.parameters(), lr=CFG.LEARNING_RATE, weight_decay=1.0e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS, eta_min=CFG.ETA_MIN)

        # ---- best model ----
        best_val_f1      = -1.0
        best_epoch       = 0
        best_state       = None
        patience         = 10
        patience_counter = 0

        pf(f"Early stopping patience: {patience} epochs (monitoring val macro F1)\n")

        for epoch in range(CFG.EPOCHS):

            train_loss = train_epoch(model, train_loader, optimizer, criterion)
            val_acc, val_f1, val_c_acc, val_c_f1 = evaluate(model, val_loader)

            scheduler.step()
            lr = scheduler.get_last_lr()[0]

            is_better = val_f1 > best_val_f1 + 1e-5

            pf(f"Epoch {epoch+1:>3}/{CFG.EPOCHS} | "
               f"LR: {lr:.2e} | Loss: {train_loss:.4f} | "
               f"LabelAcc: {val_acc:.4f} | LabelF1: {val_f1:.4f} | "
               f"ConceptAcc: {val_c_acc:.4f} | ConceptF1: {val_c_f1:.4f} "
               + (" ← best" if is_better else ""))

            if is_better:
                best_val_f1      = val_f1
                best_epoch       = epoch + 1
                best_state       = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= patience:
                pf(f"Early stopping triggered after {epoch+1} epochs")
                break

        # ---- restore best model & evaluate on test ----
        if best_state is not None:
            pf(f"\nRestoring best model from epoch {best_epoch} (val F1 = {best_val_f1:.4f})")
            model.load_state_dict(best_state)
        else:
            pf("Warning: No improvement during training — using final model")

        test_acc, test_f1, test_c_acc, test_c_f1 = evaluate(model, test_loader)

        pf(f"\n{'='*70}")
        pf(f"TEST  |  {backbone_name}")
        pf(f"Best epoch : {best_epoch}  (val F1={best_val_f1:.4f})")
        pf(f"  Label Accuracy  : {test_acc:.4f}")
        pf(f"  Label F1        : {test_f1:.4f}")
        pf(f"  Concept Accuracy: {test_c_acc:.4f}")
        pf(f"  Concept F1      : {test_c_f1:.4f}")
        pf(f"{'='*70}\n")

        # ---- checkpoint ----
        torch.save(
            {
                "model_state_dict": best_state if best_state is not None else model.state_dict(),
                "backbone":         backbone_name,
                "concept_info":     concept_info,
                "label_encoders":   label_encoders,
                "best_epoch":       best_epoch,
                "best_val_f1":      best_val_f1,
                "config": {
                    "tunable_layers":    CFG.TUNABLE_LAYERS,
                    "concept_weighting": CFG.CONCEPT_WEIGHTING,
                    "batch_size":        CFG.BATCH_SIZE,
                    "epochs_actual":     epoch + 1,
                },
                "test_results": {
                    "accuracy":         test_acc,
                    "f1":               test_f1,
                    "concept_accuracy": test_c_acc,
                    "concept_f1":       test_c_f1,
                },
            },
            os.path.join(CFG.OUTPUT_DIR, f"{tag}_best.pth"),
        )

        return {
            "run_id":                run_id,
            "backbone":              backbone_name,
            "best_epoch":            best_epoch,
            "best_val_f1":           round(best_val_f1, 4),
            "test_accuracy":         round(test_acc,   4),
            "test_f1":               round(test_f1,    4),
            "test_concept_acc":      round(test_c_acc, 4),
            "test_concept_f1":       round(test_c_f1,  4)
        }

    finally:
        log_file.close()
        original_print(f"  Log saved → {log_path}")

# ============================================================
# RUN
# ============================================================

meta = pd.read_csv(os.path.join(CFG.META_DIR, "meta.csv"))
meta[CFG.TARGET_COLUMN] = (
    meta[CFG.TARGET_COLUMN].astype(str).str.lower()
    .apply(lambda x: 1 if "melanoma" in x else 0)
)
concept_info, label_encoders = build_global_concept_encoders(meta)

backbone_results = []

for run_id, backbone_name in enumerate(BACKBONE_LIST):
    original_print(f"\n{'─'*70}")
    original_print(f"Backbone {run_id + 1}/{len(BACKBONE_LIST)}: {backbone_name}")
    original_print(f"{'─'*70}")
    result = run_backbone(backbone_name, run_id, meta, concept_info, label_encoders)
    backbone_results.append(result)